In [1]:
using JLD2
using Printf
using QuadGK
using Random
using StaticArrays
using Statistics
using WaveGreen2D.Chebyshev: ChebyshevSeries, TransformedChebyshevSeries, gradient, hessian, contains

In [2]:
data_dir = joinpath(dirname(@__DIR__), "test", "data")
mkpath(data_dir)

l1_file = joinpath(data_dir, "l1_test_data.jld2")
l2_file = joinpath(data_dir, "l2_test_data.jld2")


Random.seed!(18)
tol = eps()
imax = 1e4
qorder = 26


function mod_quadgk(f, a, b; rtol=sqrt(eps()), atol=0, maxevals=10^7, order=7)
    # Put 26 as the first try.
    qorder_vals = [[26, 25, 24]; collect(27:34)]
    
    if !(order in qorder_vals)
        pushfirst!(qorder_vals, order)
    end

    ∫f = err = nothing
    
    for qo in qorder_vals
        ∫f, err, count = quadgk_count(f, a, b; rtol=rtol, atol=atol, maxevals=maxevals, order=qo)
        if count < maxevals
            return ∫f, err
        end
    end

    @warn "Reached the maximum number of function evaluations" #maxlog=1
    
    return ∫f, err
end


function L₁(x::AbstractVector{<:Real})
    A, B, H = x
    
    f(u) = (u + H) / (u - H - (u + H)*exp(-2u))
    f₃(u) = 2 / (u - H - (u + H)*exp(-2u))^2
    f₃₃(u) = 4*(1 + exp(-2u)) / (u - H - (u + H)*exp(-2u))^3
    
    g(u) = exp(-u*(2+B)) + exp(-u*(2-B))
    g₂(u) = exp(-u*(2-B)) - exp(-u*(2+B))
    g₂₂(u) = exp(-u*(2+B)) + exp(-u*(2-B))
    
    h(u) = cos(u*A)
    h₁(u) = -sin(u*A)
    h₁₁(u) = -cos(u*A)
    
    p(u) = (f(u) * g(u) * h(u) + exp(-u)) / u

    p₁(u) = f(u) * g(u) * h₁(u)
    p₂(u) = f(u) * g₂(u) * h(u)
    p₃(u) = f₃(u) * g(u) * h(u)

    p₁₁(u) = f(u) * g(u) * h₁₁(u) * u
    p₁₂(u) = f(u) * g₂(u) * h₁(u) * u
    p₁₃(u) = f₃(u) * g(u) * h₁(u) * u

    p₂₁(u) = f(u) * g₂(u) * h₁(u) * u
    p₂₂(u) = f(u) * g₂₂(u) * h(u) * u
    p₂₃(u) = f₃(u) * g₂(u) * h(u) * u

    p₃₁(u) = f₃(u) * g(u) * h₁(u) * u
    p₃₂(u) = f₃(u) * g₂(u) * h(u) * u
    p₃₃(u) = f₃₃(u) * g(u) * h(u)
    
    path = (0.0, H+im, H+1.0, Inf)

    y = 0.0
    y₁, y₂, y₃ = [0.0 for _ in 1:3]
    y₁₁, y₁₂, y₁₃, y₂₂, y₂₃, y₃₃ = [0.0 for _ in 1:6]

    for i in 1:length(path)-1
        y += mod_quadgk(p, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        
        y₁ += mod_quadgk(p₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂ += mod_quadgk(p₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃ += mod_quadgk(p₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]

        y₁₁ += mod_quadgk(p₁₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₂ += mod_quadgk(p₁₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₃ += mod_quadgk(p₁₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₂ += mod_quadgk(p₂₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₃ += mod_quadgk(p₂₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃₃ += mod_quadgk(p₃₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
    end

    y = real(y)
    
    ∇y = real(SA[y₁, y₂, y₃])
    
    Hy = real(SA[y₁₁ y₁₂ y₁₃;
                 y₁₂ y₂₂ y₂₃;
                 y₁₃ y₂₃ y₃₃])
    
    return y, ∇y, Hy
end


function L₂(x::AbstractVector{<:Real})
    A, B, H = x
    
    # Auxilary variables for f, g and its derivatives
    a(u) = u+H
    b(u) = u-H
    c(u) = (b(u) - a(u)*exp(-2u))
    d(u) = b(u)^2 - a(u)*b(u)*exp(-2u)
    e(u) = b(u)^2 + d(u)
    dH(u) = -2*(b(u) - H*exp(-2u))
    
    f(u)   = a(u) / c(u)
    f₃(u)  = 2 / c(u)^2
    f₃₃(u) = 4*(1 + exp(-2u)) / c(u)^3
    
    g(u)   = a(u)^2 / d(u)
    g₃(u)  = 2*a(u) * (b(u)^2 + d(u)) / (b(u) * d(u)^2)
    g₃₃(u) = 2*(-2a(u)*b(u)*e(u)*dH(u) + a(u)*d(u)*e(u) + b(u)*d(u)*(e(u) - a(u)*(2b(u)-dH(u)))) / (b(u)^2 * d(u)^3)
    
    p(u)   = exp(-u*(2+B))
    p₂(u)  = -exp(-u*(2+B))
    p₂₂(u) = u*exp(-u*(2+B))
    
    q(u)   = exp(-u*(4-B))
    q₂(u)  = exp(-u*(4-B))
    q₂₂(u) = u*exp(-u*(4-B))
    
    r(u)   = cos(u*A)
    r₁(u)  = -sin(u*A)
    r₁₁(u) = -u*cos(u*A)
    
    h(u) = (f(u) * p(u) + g(u) * q(u)) * r(u) / u
    
    h₁(u) = (f(u) * p(u) + g(u) * q(u)) * r₁(u)
    h₂(u) = (f(u) * p₂(u) + g(u) * q₂(u)) * r(u)
    h₃(u) = (f₃(u) * p(u) + g₃(u) * q(u)) * r(u)
    
    h₁₁(u) = (f(u) * p(u) + g(u) * q(u)) * r₁₁(u)
    h₁₂(u) = (f(u) * p₂(u) + g(u) * q₂(u)) * r₁(u) * u
    h₁₃(u) = (f₃(u) * p(u) + g₃(u) * q(u)) * r₁(u) * u
    h₂₂(u) = (f(u) * p₂₂(u) + g(u) * q₂₂(u)) * r(u)
    h₂₃(u) = (f₃(u) * p₂(u) + g₃(u) * q₂(u)) * r(u) * u
    h₃₃(u) = (f₃₃(u) * p(u) + g₃₃(u) * q(u)) * r(u)
    
    path = (0.0, H+im, H+1.0, Inf)

    y = 0.0
    y₁, y₂, y₃ = [0.0 for _ in 1:3]
    y₁₁, y₁₂, y₁₃, y₂₂, y₂₃, y₃₃ = [0.0 for _ in 1:6]

    for i in 1:length(path)-1
        y += mod_quadgk(h, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        
        y₁ += mod_quadgk(h₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂ += mod_quadgk(h₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃ += mod_quadgk(h₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]

        y₁₁ += mod_quadgk(h₁₁, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₂ += mod_quadgk(h₁₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₁₃ += mod_quadgk(h₁₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₂ += mod_quadgk(h₂₂, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₂₃ += mod_quadgk(h₂₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
        y₃₃ += mod_quadgk(h₃₃, path[i], path[i+1]; rtol=tol, atol=tol, maxevals=imax, order=qorder)[1]
    end

    y = real(y)
    
    ∇y = real(SA[y₁, y₂, y₃])
    
    Hy = real(SA[y₁₁ y₁₂ y₁₃;
                 y₁₂ y₂₂ y₂₃;
                 y₁₃ y₂₃ y₃₃])
    
    return y, ∇y, Hy
end;

In [3]:
@load "../src/chebyshev_series.jld2" L₁_series L₂_series;

In [4]:
function u(x::SVector{3,Float64})
    A, B, H = x

    return SA[A, B, log(H)]
end


function ∇u(x::SVector{3,Float64})
    _, _, H = x

    grad_u = one(MMatrix{3, 3, Float64})
    grad_u[3, 3] = 1/H
    
    return SMatrix(grad_u)
end


function Hu(x::SVector{3,Float64})
    _, _, H = x

    hess_u = zero(MArray{Tuple{3, 3, 3}, Float64, 3})
    hess_u[3, 3, 3] = -1/H^2

    return SArray(hess_u)
end;

In [5]:
C₁1 = L₁_series[1]
C₁2 = TransformedChebyshevSeries(L₁_series[2], u, ∇u, Hu)

C₂1 = TransformedChebyshevSeries(L₂_series[1], u, ∇u, Hu)
C₂2 = TransformedChebyshevSeries(L₂_series[2], u, ∇u, Hu)
C₂3 = TransformedChebyshevSeries(L₂_series[3], u, ∇u, Hu)

function C₁(x::SVector{3, Float64})
    if contains(C₁1, x)
        return C₁1(x)
    elseif contains(C₁2, x)
        return C₁2(x)
    else
        throw(DomainError(x))
    end
end


function C₁_gradient(x::SVector{3, Float64})
    if contains(C₁1, x)
        return gradient(C₁1, x)
    elseif contains(C₁2, x)
        return gradient(C₁2, x)
    else
        throw(DomainError(x))
    end
end


function C₁_hessian(x::SVector{3, Float64})
    if contains(C₁1, x)
        return hessian(C₁1, x)
    elseif contains(C₁2, x)
        return hessian(C₁2, x)
    else
        throw(DomainError(x))
    end
end


function C₂(x::SVector{3, Float64})
    if contains(C₂1, x)
        return C₂1(x)
    elseif contains(C₂2, x)
        return C₂2(x)
    elseif contains(C₂3, x)
        return C₂3(x)
    else
        throw(DomainError(x))
    end
end


function C₂_gradient(x::SVector{3, Float64})
    if contains(C₂1, x)
        return gradient(C₂1, x)
    elseif contains(C₂2, x)
        return gradient(C₂2, x)
    elseif contains(C₂3, x)
        return gradient(C₂3, x)
    else
        throw(DomainError(x))
    end
end


function C₂_hessian(x::SVector{3, Float64})
    if contains(C₂1, x)
        return hessian(C₂1, x)
    elseif contains(C₂2, x)
        return hessian(C₂2, x)
    elseif contains(C₂3, x)
        return hessian(C₂3, x)
    else
        throw(DomainError(x))
    end
end;

In [6]:
x = SA[0.1, 0.5, 0.3]

y₁, ∇y₁, Hy₁ = L₁(x)
y₂, ∇y₂, Hy₂ = L₂(x)

println(C₁(x) - y₁)
println(C₂(x) - y₂)

z₁, ∇z₁ = C₁_gradient(x)
z₂, ∇z₂ = C₂_gradient(x)

println(z₁ - y₁)
println(z₂ - y₂)
println(maximum(abs.(∇z₁ - ∇y₁)))
println(maximum(abs.(∇z₂ - ∇y₂)))

z₁, ∇z₁, Hz₁ = C₁_hessian(x)
z₂, ∇z₂, Hz₂ = C₂_hessian(x)

println(z₁ - y₁)
println(z₂ - y₂)
println(maximum(abs.(∇z₁ - ∇y₁)))
println(maximum(abs.(∇z₂ - ∇y₂)))
println(maximum(abs.(Hz₁ - Hy₁)))
println(maximum(abs.(Hz₂ - Hy₂)))

1.5543122344752192e-15
-4.440892098500626e-16
1.5543122344752192e-15
-4.440892098500626e-16
2.886579864025407e-14
7.460698725481052e-14
1.5543122344752192e-15
-4.440892098500626e-16
2.886579864025407e-14
7.460698725481052e-14
5.370565103746117e-13
2.0605739337042905e-13


In [7]:
x = SA[0.25, 0.8, 1.1]

y₁, ∇y₁, Hy₁ = L₁(x)
y₂, ∇y₂, Hy₂ = L₂(x)

println(C₁(x) - y₁)
println(C₂(x) - y₂)

z₁, ∇z₁ = C₁_gradient(x)
z₂, ∇z₂ = C₂_gradient(x)

println(z₁ - y₁)
println(z₂ - y₂)
println(maximum(abs.(∇z₁ - ∇y₁)))
println(maximum(abs.(∇z₂ - ∇y₂)))

z₁, ∇z₁, Hz₁ = C₁_hessian(x)
z₂, ∇z₂, Hz₂ = C₂_hessian(x)

println(z₁ - y₁)
println(z₂ - y₂)
println(maximum(abs.(∇z₁ - ∇y₁)))
println(maximum(abs.(∇z₂ - ∇y₂)))
println(maximum(abs.(Hz₁ - Hy₁)))
println(maximum(abs.(Hz₂ - Hy₂)))

0.0
3.885780586188048e-16
0.0
3.885780586188048e-16
1.2032042029375134e-14
1.5307199952019346e-14
0.0
3.885780586188048e-16
1.2032042029375134e-14
1.5307199952019346e-14
5.795364188543317e-13
1.8141044222375058e-13


In [8]:
x = SA[0.3, 0.9, 2.0]

y₁, ∇y₁, Hy₁ = L₁(x)
y₂, ∇y₂, Hy₂ = L₂(x)

println(C₁(x) - y₁)
println(C₂(x) - y₂)

z₁, ∇z₁ = C₁_gradient(x)
z₂, ∇z₂ = C₂_gradient(x)

println(z₁ - y₁)
println(z₂ - y₂)
println(maximum(abs.(∇z₁ - ∇y₁)))
println(maximum(abs.(∇z₂ - ∇y₂)))

z₁, ∇z₁, Hz₁ = C₁_hessian(x)
z₂, ∇z₂, Hz₂ = C₂_hessian(x)

println(z₁ - y₁)
println(z₂ - y₂)
println(maximum(abs.(∇z₁ - ∇y₁)))
println(maximum(abs.(∇z₂ - ∇y₂)))
println(maximum(abs.(Hz₁ - Hy₁)))
println(maximum(abs.(Hz₂ - Hy₂)))

3.3306690738754696e-16
-7.216449660063518e-16
3.3306690738754696e-16
-7.216449660063518e-16
2.3925306180672123e-14
1.2004286453759505e-15
3.3306690738754696e-16
-7.216449660063518e-16
2.3925306180672123e-14
1.2004286453759505e-15
9.57900425646585e-13
1.5712431356007528e-13


In [2]:
@load "../test/data/l1_test_data.jld2" L1 ∇L1 HL1
@load "../test/data/l2_test_data.jld2" L2 ∇L2 HL2

In [3]:
Lc = Vector{Float64}(undef, size(xv, 1))
∇Lc = Matrix{Float64}(undef, size(xv))
HLc = Array{Float64, 3}(undef, size(xv)..., 3);

In [4]:
i = 1
Lc1 = L₁c(view(xv, i, :))
Lc2, ∇Lc1 = gradient(L₁c, view(xv, i, :))
Lc3, ∇Lc2, HL1 = hessian(L₁c, view(xv, i, :));

In [5]:
for i in axes(xv, 1)
    Lc1 = L₁c(view(xv, i, :))
    Lc2, ∇Lc1 = gradient(L₁c, view(xv, i, :))
    Lc3, ∇Lc2, HL1 = hessian(L₁c, view(xv, i, :))

    if Lc1 ≈ Lc2 && Lc1 ≈ Lc3
        Lc[i] = Lc1
    end

    if ∇Lc1 ≈ ∇Lc2
        ∇Lc[i, :] = ∇Lc1
    end
    
    HLc[i, :, :] = HL1
end;

In [6]:
err_L = @. abs(L - Lc)
err_∇L = @. abs(∇L - ∇Lc)
err_HL = @. abs(HL - HLc)

mean_err_L = mean(err_L)
mean_err_∇L = dropdims(mean(err_∇L; dims=1); dims=1)
mean_err_HL = dropdims(mean(err_HL; dims=1); dims=1)

std_err_L = std(err_L)
std_err_∇L = dropdims(std(err_∇L; dims=1); dims=1)
std_err_HL = dropdims(std(err_HL; dims=1); dims=1)

lim_err_L = mean_err_L + 3 * std_err_L
lim_err_∇L = reshape(mean_err_∇L + 3 * std_err_∇L, 1, 3)
lim_err_HL = reshape(mean_err_HL + 3 * std_err_HL, 1, 3, 3)

pct_err_L = sum(err_L .< lim_err_L) * 100 / size(xv, 1)
pct_err_∇L = dropdims(count(err_∇L .< lim_err_∇L; dims=1); dims=1) * 100 / size(xv, 1)
pct_err_HL = dropdims(sum(err_HL .< lim_err_HL; dims=1); dims=1) * 100 / size(xv, 1);

In [7]:
result = """
L₁c mean absolute error = $(@sprintf("%.0e", mean_err_L))

∇L₁c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_∇L...))

HL₁c mean absolute error = $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[1, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[2, :]...))
                           $(@sprintf("[%.0e %6.0e %6.0e]", mean_err_HL[3, :]...))

For L₁c, ∇L₁c and HL₁c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₁c error values lower than reference = $(@sprintf("%.1f%%", pct_err_L))

Percentage of ∇L₁c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_∇L...))

Percentage of HL₁c error values lower than reference = $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[1, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[2, :]...))
                                                       $(@sprintf("[%.1f%% %5.1f%% %5.1f%%]", pct_err_HL[3, :]...))
"""

println(result)

L₁c mean absolute error = 6e-16

∇L₁c mean absolute error = [2e-14  1e-14  2e-14]

HL₁c mean absolute error = [2e-12  6e-13  8e-13]
                           [6e-13  8e-13  6e-13]
                           [8e-13  6e-13  3e-12]

For L₁c, ∇L₁c and HL₁c, the error value for each point is compared with a corresponding reference
value. The reference value is the mean absolute error plus three times the standard deviation. The
following results display the percentage of points for which the error is less than the reference.

Percentage of L₁c error values lower than reference = 98.9%

Percentage of ∇L₁c error values lower than reference = [98.6%  98.7%  98.9%]

Percentage of HL₁c error values lower than reference = [98.7%  99.0%  98.8%]
                                                       [99.0%  98.9%  98.8%]
                                                       [98.8%  98.8%  99.5%]

